# Mokka Decisions — Kaggle training notebook

Installs the pinned project wheel from the private bundle dataset and runs the
**same trainer** used locally (`mokka_decisions.train.run_training`). There is no
second training implementation.

Requirements: attach the `gjusev/mokka-decisions-bundle` dataset (wheel + configs
+ prepared data), GPU T4, internet enabled. Set `STAGE` below (`train`,
`resume_test`, `eval`, `baselines`, `smoke`).

In [ ]:
STAGE = 'train'
CONFIG = 'configs/pilot.yaml'  # or configs/train.yaml for the full run

from pathlib import Path
bundle = next(Path('/kaggle/input').rglob('BUNDLE_MANIFEST.json')).parent
print('bundle:', bundle)
!pip install -q --no-deps "$bundle/dist/"mokka_decisions-*.whl
!pip install -q transformers==5.19.0 datasets==5.1.0 safetensors sentencepiece
import torch; print('gpu:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# run the shared job script with the selected stage (same code path as the kernel script)
import subprocess, sys, urllib.request
src = bundle.parent.parent  # not used; fetch run.py from the attached kernel sources if present
run_py = Path('/kaggle/input')
candidates = list(run_py.rglob('run.py'))
assert candidates, 'attach the kernel source or copy run.py next to this notebook'
cmd = [sys.executable, str(candidates[0]), '--stage', STAGE, '--config', CONFIG]
print(' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
import json
from pathlib import Path
status = json.loads(Path('/kaggle/working/run_status.json').read_text())
print(json.dumps(status, indent=2))
# verify: a finished train stage must report final_global_step > 0 and a checkpoint file
if status.get('stage') == 'train':
    assert status['final_global_step'] > 0
    assert any(Path('/kaggle/working').rglob('*.safetensors'))
    print('verified completed training run')